# A Catholic Introduction to Artificial Intelligence
## Final Project — Module 4: Bias Auditing & Fairness Analysis
### Classes 7 & 8

---

> *"When AI systems present themselves as neutral and objective, they end up reflecting and reinforcing the stereotypes or ideological bias of their designers and developers... The exclusion of the vulnerable becomes cloaked in a veneer of neutrality and objectivity, against which it becomes difficult to raise objections. In this way, injustice goes unnoticed."*
> — Pope Leo XIV, *Magnifica Humanitas*, no. 102–103

---

## Module Overview

In Module 3, you discovered a striking problem: a model that is 81% accurate overall catches only about 6% of the late submissions it was designed to identify. This module goes deeper — investigating whether the model's errors are distributed **equally** across all students and assignment types, or whether they concentrate disproportionately on particular groups.

This is called a **bias audit**. It is one of the most important steps in responsible AI development, and it is one that many real-world AI systems skip entirely.

A model that is 90% accurate for some groups and 60% accurate for others may look fine in aggregate statistics while causing serious harm to the groups it serves worst. Those groups are often — not coincidentally — the groups that were already underrepresented in the training data or disadvantaged in the real world.

### What You Will Do in This Module

1. Rebuild and fully document the model pipeline from Module 3
2. Conduct a structured bias audit across multiple dimensions
3. Measure performance disparities quantitatively
4. Investigate the root causes of those disparities
5. Evaluate what the disparities mean ethically — and what, if anything, can be done about them

---

## Setup: Full Pipeline from Module 3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)

# ── Load and prepare ──────────────────────────────────────────────────
df = pd.read_csv('synthetic_homework_dataset.csv',
                 parse_dates=['date_assigned', 'date_submitted'])

df['time_pressure'] = df['difficulty'] / df['days_until_due']
le = LabelEncoder()
df['assignment_type_enc'] = le.fit_transform(df['assignment_type'])
print('Assignment type encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

FULL_FEATURES = [
    'num_questions', 'difficulty', 'days_until_due', 'time_pressure',
    'assignment_type_enc', 'prior_completion_rate',
    'prior_avg_grade', 'prior_avg_homework_time'
]
CONSERVATIVE_FEATURES = [
    'num_questions', 'difficulty', 'days_until_due',
    'time_pressure', 'assignment_type_enc'
]

TARGET = 'completed_on_time'

# ── Train the model (exact replication of Module 3) ───────────────────
X = df[FULL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

# ── Build a rich test dataframe with all original columns ─────────────
results = df.loc[X_test.index].copy()
results['predicted']    = y_pred
results['prob_on_time'] = y_prob.round(4)
results['correct']      = (results['predicted'] == results[TARGET]).astype(int)
results['error_type']   = 'Correct'
results.loc[(results['predicted'] == 1) & (results[TARGET] == 0), 'error_type'] = 'Missed late'
results.loc[(results['predicted'] == 0) & (results[TARGET] == 1), 'error_type'] = 'Wrongly flagged'

overall_acc = accuracy_score(y_test, y_pred)
print(f'\nModel rebuilt. Overall test accuracy: {overall_acc*100:.1f}%')
print(f'Test set size: {len(results)}')
print(f'Error breakdown: {results["error_type"].value_counts().to_dict()}')

---

## Part 1: What Is a Bias Audit?

A bias audit asks: **does the model perform equally well for all subgroups?**

"Performance" means different things in different contexts:
- **Overall accuracy** — what fraction of predictions are correct?
- **Late catch rate (recall)** — of all truly late submissions, what fraction does the model identify?
- **False flag rate** — of all on-time submissions, what fraction does the model wrongly label as late?
- **Average predicted probability** — does the model express different levels of confidence for different groups?

A model can be accurate overall while being systematically **inaccurate** for a specific subgroup. When that subgroup is already disadvantaged, the model compounds existing inequality rather than reducing it.

### Audit Dimensions

We will audit the model across four dimensions:

1. **Assignment type** — does the model work equally well for essays, coding tasks, reading, etc.?
2. **Difficulty level** — is the model less accurate for hard assignments?
3. **Prior completion rate tier** — does the model treat students with weaker histories differently?
4. **Individual students** — are some students consistently mislabeled?

Each dimension reveals something different about the model's failure modes.

In [ ]:
# Helper function: compute audit metrics for any subgroup
def audit_group(subset, group_name='Group'):
    """Returns a dict of fairness metrics for a subset of the test data."""
    n = len(subset)
    if n == 0:
        return None

    actual_late  = (subset[TARGET] == 0).sum()
    actual_ontime = (subset[TARGET] == 1).sum()

    caught_late   = ((subset['predicted'] == 0) & (subset[TARGET] == 0)).sum()
    missed_late   = ((subset['predicted'] == 1) & (subset[TARGET] == 0)).sum()  # FP
    wrongly_flagged = ((subset['predicted'] == 0) & (subset[TARGET] == 1)).sum()  # FN

    accuracy     = subset['correct'].mean()
    late_recall  = caught_late / actual_late if actual_late > 0 else np.nan
    false_flag_rate = wrongly_flagged / actual_ontime if actual_ontime > 0 else np.nan
    avg_prob     = subset['prob_on_time'].mean()

    return {
        'Group': group_name,
        'n': n,
        'Actual late': actual_late,
        'Accuracy': round(accuracy, 3),
        'Late recall': round(late_recall, 3) if not np.isnan(late_recall) else 'N/A',
        'False flag rate': round(false_flag_rate, 3) if not np.isnan(false_flag_rate) else 'N/A',
        'Avg predicted prob': round(avg_prob, 3)
    }

print('Audit helper function defined.')
print('It measures: accuracy, late recall, false flag rate, and avg predicted probability.')

---

## Part 2: Audit by Assignment Type

Does the model perform equally well for all types of homework? A model that works well for worksheets but poorly for essays might be reinforcing existing patterns about which kinds of academic work students struggle with.

In [ ]:
# Audit across assignment types
type_audit = []
for atype in sorted(results['assignment_type'].unique()):
    subset = results[results['assignment_type'] == atype]
    row = audit_group(subset, atype)
    if row:
        type_audit.append(row)

# Add overall row
overall = audit_group(results, 'OVERALL')
type_audit.append(overall)

type_df = pd.DataFrame(type_audit)
print('=== Audit by Assignment Type ===')
print(type_df.to_string(index=False))

In [ ]:
# Visualize accuracy and error types by assignment type
type_counts = results.groupby(['assignment_type', 'error_type']).size().unstack(fill_value=0)
# Ensure all columns exist
for col in ['Correct', 'Missed late', 'Wrongly flagged']:
    if col not in type_counts.columns:
        type_counts[col] = 0
type_counts = type_counts[['Correct', 'Missed late', 'Wrongly flagged']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked error bars
type_counts.plot(
    kind='bar', stacked=True, ax=axes[0],
    color=['#4CAF50', '#E57373', '#FFB74D'],
    edgecolor='white'
)
axes[0].set_title('Prediction Outcomes by Assignment Type', fontweight='bold')
axes[0].set_xlabel('Assignment Type')
axes[0].set_ylabel('Number of Predictions')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(loc='upper right', fontsize=9)

# Right: accuracy by type with overall reference line
acc_by_type = results.groupby('assignment_type')['correct'].mean().sort_values()
colors = ['#E57373' if v < overall_acc else '#4CAF50' for v in acc_by_type.values]
axes[1].barh(acc_by_type.index, acc_by_type.values * 100, color=colors)
axes[1].axvline(overall_acc * 100, color='black', linestyle='--', linewidth=1.5,
                label=f'Overall: {overall_acc*100:.1f}%')
axes[1].set_xlabel('Accuracy (%)')
axes[1].set_title('Accuracy by Assignment Type\n(Red = below average)', fontweight='bold')
axes[1].set_xlim(0, 115)
for i, (_, v) in enumerate(acc_by_type.items()):
    axes[1].text(v * 100 + 0.5, i, f'{v*100:.1f}%', va='center', fontsize=9)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### Finding 1: Worksheet Accuracy Is Perfect — But for the Wrong Reason

The model achieved 100% accuracy on worksheets in the test set. This sounds impressive until you look more carefully: in the test set, there were **no late worksheet submissions**. The model predicts "on time" for almost everything, so it happens to be correct for every worksheet.

This illustrates a general principle: **high accuracy on a subgroup can be meaningless if that subgroup rarely appears in the minority class**. A model that never catches anyone being late will score 100% accuracy on any group where nobody is ever late — without having learned anything.

This is exactly the deceptive nature of accuracy as a metric that Module 3 introduced.

> 🤔 **Think about it:** Reading assignments have the highest number of "missed late" errors. What might explain this? Is reading a type of assignment where students are more likely to procrastinate — and if so, why might the model fail to detect it?

---

## Part 3: Audit by Difficulty Level

Does the model handle easy and hard assignments differently? This matters because if the model is less accurate for the hardest assignments, it may be least useful precisely when students are most at risk.

In [ ]:
# Audit across difficulty levels
diff_audit = []
for diff in sorted(results['difficulty'].unique()):
    subset = results[results['difficulty'] == diff]
    row = audit_group(subset, f'Difficulty {diff}')
    if row:
        diff_audit.append(row)

diff_df = pd.DataFrame(diff_audit)
print('=== Audit by Difficulty Level ===')
print(diff_df.to_string(index=False))

In [ ]:
# Visualize difficulty audit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

diff_counts = results.groupby(['difficulty', 'error_type']).size().unstack(fill_value=0)
for col in ['Correct', 'Missed late', 'Wrongly flagged']:
    if col not in diff_counts.columns:
        diff_counts[col] = 0
diff_counts = diff_counts[['Correct', 'Missed late', 'Wrongly flagged']]

diff_counts.plot(
    kind='bar', stacked=True, ax=axes[0],
    color=['#4CAF50', '#E57373', '#FFB74D'], edgecolor='white'
)
axes[0].set_title('Prediction Outcomes by Difficulty Level', fontweight='bold')
axes[0].set_xlabel('Difficulty (1=Easy, 5=Hard)')
axes[0].set_ylabel('Number of Predictions')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(fontsize=9)

acc_by_diff = results.groupby('difficulty')['correct'].mean()
late_by_diff = results[results[TARGET]==0].groupby('difficulty').apply(
    lambda g: (g['predicted'] == 0).mean()
)

x = np.array(sorted(results['difficulty'].unique()))
axes[1].bar(x - 0.2, acc_by_diff.values * 100, width=0.35,
            label='Accuracy', color='#5C8BC7', alpha=0.85)
# Only plot late recall where we have late submissions in test set
late_x = [d for d in x if d in late_by_diff.index]
late_y = [late_by_diff[d] * 100 for d in late_x]
axes[1].bar(np.array(late_x) + 0.2, late_y, width=0.35,
            label='Late catch rate', color='#E57373', alpha=0.85)
axes[1].axhline(overall_acc * 100, color='navy', linestyle='--',
                linewidth=1, label=f'Overall accuracy: {overall_acc*100:.1f}%')
axes[1].set_xlabel('Difficulty Level')
axes[1].set_ylabel('%')
axes[1].set_title('Accuracy vs. Late Catch Rate by Difficulty', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{d}' for d in x])
axes[1].set_ylim(0, 120)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### Finding 2: Perfect Accuracy on Difficulty 3 — But No Late Students to Catch

The same pattern repeats: the model scores perfectly on difficulty-3 assignments because there are no late submissions in the test set for that difficulty level. For difficulty 5 (hardest) assignments, the model has the most late submissions to catch — and catches only 1 out of 5.

This means the model is **least useful precisely when the stakes are highest** — when students are attempting the hardest material and most likely to fall behind.

> 📝 **Reflection:** If a system is most likely to fail for the students facing the hardest work, what does that mean for who the system actually serves? How does this relate to Catholic social teaching's preferential option for those who are most vulnerable?

*[Write your answer here]*

---

## Part 4: The Most Important Audit — Prior Completion Rate Tier

This is the bias audit that matters most. We categorized students into four tiers based on their historical completion rate. Let's examine whether the model performs equally for students in each tier.

In [ ]:
# Create completion rate tiers
results['prior_tier'] = pd.cut(
    results['prior_completion_rate'],
    bins=[0, 0.60, 0.75, 0.90, 1.01],
    labels=['<60%', '60–75%', '75–90%', '>90%']
)
df['prior_tier'] = pd.cut(
    df['prior_completion_rate'],
    bins=[0, 0.60, 0.75, 0.90, 1.01],
    labels=['<60%', '60–75%', '75–90%', '>90%']
)

# Full dataset late rate by tier — what is the ground truth?
full_tier_stats = df.groupby('prior_tier', observed=True).agg(
    n=('completed_on_time', 'count'),
    actual_late_rate=('completed_on_time', lambda x: 1 - x.mean()),
    avg_prior_grade=('prior_avg_grade', 'mean')
).reset_index()

print('=== Ground Truth: Late Rate by Prior Completion Tier (full dataset) ===')
print(full_tier_stats.round(3).to_string(index=False))
print()
print('Students with lower prior completion rates are MUCH more likely to be late.')
print('This is the pattern the model has learned — and the pattern that creates the bias.')

In [ ]:
# Audit by completion rate tier
tier_order = ['<60%', '60–75%', '75–90%', '>90%']
tier_audit = []
for tier in tier_order:
    subset = results[results['prior_tier'] == tier]
    row = audit_group(subset, tier)
    if row:
        tier_audit.append(row)

tier_df = pd.DataFrame(tier_audit)
print('=== Model Performance by Prior Completion Rate Tier ===')
print(tier_df.to_string(index=False))
print()

# Highlight the disparity
accs = [r['Accuracy'] for r in tier_audit if r]
print(f'Accuracy range across tiers: {min(accs):.3f} to {max(accs):.3f}')
print(f'Accuracy gap: {max(accs) - min(accs):.3f} ({(max(accs)-min(accs))*100:.1f} percentage points)')

In [ ]:
# Visualize the performance disparity across tiers
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Accuracy and actual late rate by tier
tier_counts = results.groupby(['prior_tier', 'error_type'], observed=True).size().unstack(fill_value=0)
for col in ['Correct', 'Missed late', 'Wrongly flagged']:
    if col not in tier_counts.columns:
        tier_counts[col] = 0
tier_counts = tier_counts[['Correct', 'Missed late', 'Wrongly flagged']]

tier_counts.plot(
    kind='bar', stacked=True, ax=axes[0],
    color=['#4CAF50', '#E57373', '#FFB74D'], edgecolor='white'
)
axes[0].set_title('Prediction Outcomes by\nPrior Completion Rate Tier', fontweight='bold')
axes[0].set_xlabel('Prior Completion Rate Tier')
axes[0].set_ylabel('Number of Predictions')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(fontsize=9)

# Right: Accuracy vs. actual late rate — the gap that reveals the problem
x = np.arange(len(tier_order))
test_acc_by_tier = results.groupby('prior_tier', observed=True)['correct'].mean()
test_late_by_tier = df.groupby('prior_tier', observed=True)['completed_on_time'].apply(
    lambda s: 1 - s.mean()
)

acc_vals = [test_acc_by_tier.get(t, 0) * 100 for t in tier_order]
late_vals = [test_late_by_tier.get(t, 0) * 100 for t in tier_order]

axes[1].bar(x - 0.2, acc_vals, width=0.35,
            label='Model accuracy', color='#5C8BC7', alpha=0.9)
axes[1].bar(x + 0.2, late_vals, width=0.35,
            label='Actual late rate', color='#E57373', alpha=0.9)
axes[1].axhline(overall_acc * 100, color='navy', linestyle='--',
                linewidth=1, label=f'Overall accuracy: {overall_acc*100:.1f}%')
axes[1].set_xticks(x)
axes[1].set_xticklabels(tier_order)
axes[1].set_xlabel('Prior Completion Rate Tier')
axes[1].set_ylabel('%')
axes[1].set_title('Model Accuracy vs. Actual Late Rate\nby Prior Completion Tier', fontweight='bold')
axes[1].set_ylim(0, 115)
axes[1].legend(fontsize=9)

for i, (a, l) in enumerate(zip(acc_vals, late_vals)):
    axes[1].text(i - 0.2, a + 1, f'{a:.0f}%', ha='center', fontsize=8)
    axes[1].text(i + 0.2, l + 1, f'{l:.0f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

### Finding 3: A 33-Percentage-Point Accuracy Gap — And What It Means

The audit reveals a stark performance disparity:

| Tier | Accuracy | Actual late rate | What this means |
|------|----------|------------------|-----------------|
| **>90%** | 96% | 5% | Works well — but these students rarely need help |
| **75–90%** | 91% | 14% | Reasonable — but catches 0 of 5 late submissions |
| **60–75%** | 63% | 30% | Fails — catches only 1 of 7 late submissions |
| **<60%** | 50% | 60% | Near-random — even the model's confidence is uncertain |

The model performs best for students with the highest prior completion rates — students who are least likely to need any intervention at all. It performs worst for students with the lowest prior completion rates — the students who are most likely to be late and most likely to need support.

This is not an accident. It is a structural consequence of how the model was trained:

1. Students with low prior completion rates are **underrepresented** in the training data (only 5 students out of 500 records have prior rates below 60%)
2. The model has very few examples of this group to learn from
3. When it encounters these students, it has little reliable signal beyond a weak prior — and tends to predict "on time" because that's the safe default

The result: the students the system most needs to identify are the ones it is least capable of identifying.

In [ ]:
# How represented is each tier in the training data?
train_df = df.loc[X_train.index].copy()
train_df['prior_tier'] = pd.cut(
    train_df['prior_completion_rate'],
    bins=[0, 0.60, 0.75, 0.90, 1.01],
    labels=['<60%', '60–75%', '75–90%', '>90%']
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Training data representation
train_tier_counts = train_df['prior_tier'].value_counts()[tier_order]
test_tier_counts  = results['prior_tier'].value_counts().reindex(tier_order, fill_value=0)

x = np.arange(len(tier_order))
axes[0].bar(x - 0.2, train_tier_counts.values, width=0.35,
            label='Training set', color='#7986CB', alpha=0.9)
axes[0].bar(x + 0.2, test_tier_counts.values, width=0.35,
            label='Test set', color='#81C784', alpha=0.9)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tier_order)
axes[0].set_xlabel('Prior Completion Rate Tier')
axes[0].set_ylabel('Number of Records')
axes[0].set_title('Training & Test Set Representation\nby Prior Completion Tier', fontweight='bold')
axes[0].legend()
for i, (tr, te) in enumerate(zip(train_tier_counts.values, test_tier_counts.values)):
    axes[0].text(i - 0.2, tr + 0.5, str(tr), ha='center', fontsize=9)
    axes[0].text(i + 0.2, te + 0.5, str(te), ha='center', fontsize=9)

# Predicted probability distribution by tier
for tier in tier_order:
    subset = results[results['prior_tier'] == tier]['prob_on_time']
    if len(subset) > 0:
        axes[1].hist(subset, bins=12, alpha=0.55, label=f'{tier} (n={len(subset)})')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold = 0.5')
axes[1].set_xlabel('Predicted probability of on-time submission')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Predicted Probabilities\nby Prior Completion Tier', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### The Root Cause: Underrepresentation

The chart on the left reveals why the bias exists: students with prior completion rates below 75% are dramatically underrepresented in both the training and test sets. The model has seen very few examples of students who struggle consistently, so it has not learned to reliably identify them.

The chart on the right shows the result: for students in the 60–75% tier, predicted probabilities are more spread out and uncertain. The model is less confident — and less accurate — for this group.

This is a specific instance of a general problem in AI: **training data that does not represent vulnerable groups produces models that serve those groups worst.** The dataset reflects the real world — students with low completion rates are rare in it because they are rare in the population. But that rarity means the model never adequately learns their patterns.

> 🤔 **Think about it:** *Magnifica Humanitas* (no. 80) warns that in the digital age, justice demands we prevent new forms of exclusion — including algorithms that perpetuate prejudice and discrimination for underrepresented groups. Is the bias we found here intentional? Does it matter if it wasn't intentional?

---

## Part 5: Per-Student Bias Analysis

Aggregate statistics across tiers can hide individual patterns. Let's look at which specific students are consistently mislabeled.

In [ ]:
# Per-student performance on test set
student_audit = results.groupby('student_id').apply(
    lambda g: pd.Series({
        'n_predictions': len(g),
        'accuracy': g['correct'].mean(),
        'missed_late': ((g['predicted']==1) & (g[TARGET]==0)).sum(),
        'wrongly_flagged': ((g['predicted']==0) & (g[TARGET]==1)).sum(),
        'actual_late_rate': (g[TARGET]==0).mean(),
        'avg_prob_predicted': g['prob_on_time'].mean(),
        'prior_completion_rate': g['prior_completion_rate'].mean(),
        'prior_avg_grade': g['prior_avg_grade'].mean(),
    })
).reset_index()

# Students with at least 2 predictions for reliability
student_audit_filtered = student_audit[student_audit['n_predictions'] >= 2].copy()

print('=== Students with Worst Model Performance (≥2 predictions) ===')
worst = student_audit_filtered.sort_values('accuracy').head(10)
print(worst[['student_id','n_predictions','accuracy','missed_late',
              'wrongly_flagged','actual_late_rate',
              'prior_completion_rate','avg_prob_predicted']].round(3).to_string(index=False))

In [ ]:
# Scatter: prior completion rate vs. model accuracy for each student
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(
    student_audit_filtered['prior_completion_rate'],
    student_audit_filtered['accuracy'],
    c=student_audit_filtered['actual_late_rate'],
    cmap='RdYlGn', vmin=0, vmax=0.5,
    s=80, alpha=0.8, edgecolors='white', linewidths=0.5
)
plt.colorbar(sc, ax=axes[0], label='Actual late rate')
axes[0].axhline(overall_acc, color='navy', linestyle='--',
                linewidth=1, label=f'Overall: {overall_acc*100:.1f}%')
axes[0].set_xlabel('Prior Completion Rate')
axes[0].set_ylabel('Model Accuracy for This Student')
axes[0].set_title('Prior History vs. Model Accuracy\n(Color = actual late rate)', fontweight='bold')
axes[0].legend(fontsize=9)

# Histogram of per-student accuracy
axes[1].hist(student_audit_filtered['accuracy'], bins=10,
             color='#5C8BC7', edgecolor='white')
axes[1].axvline(overall_acc, color='red', linestyle='--',
                linewidth=1.5, label=f'Overall mean: {overall_acc*100:.1f}%')
axes[1].set_xlabel('Per-Student Model Accuracy')
axes[1].set_ylabel('Number of Students')
axes[1].set_title('Distribution of Model Accuracy\nAcross Individual Students', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Students where model accuracy < 70%: {(student_audit_filtered["accuracy"] < 0.70).sum()}')
print(f'Students where model accuracy = 100%: {(student_audit_filtered["accuracy"] == 1.0).sum()}')

### Finding 4: Some Students Are Almost Always Mislabeled

The scatter plot makes the pattern clear: students with lower prior completion rates (left side of the chart) tend to be the ones the model gets wrong most often. And because the model defaults to predicting "on time," the errors are almost entirely **missed late submissions** — the model fails to flag the very students who most need support.

There are students in the test set where the model is wrong on every single prediction. These are real people (or simulated ones). If this model were deployed in a school:

- The students the teacher would be told to watch out for would be different from the students actually struggling
- The students actually struggling would receive no alert — no early check-in, no additional support
- The students doing fine might receive unnecessary attention or scrutiny

The model would be, in practice, counterproductive for the students who need it most.

---

## Part 6: Comparing Feature Sets — Does Removing History Help?

In Module 2, you were asked whether to include `prior_completion_rate` and `prior_avg_grade`. Module 3 showed that `prior_completion_rate` is the most influential feature. Now we can ask the bias question directly: **does removing these history-based features reduce the performance disparity?**

In [ ]:
# Train the conservative model (no student history)
X_cons = df[CONSERVATIVE_FEATURES]

Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    X_cons, y, test_size=0.2, random_state=42, stratify=y
)
sc_c = StandardScaler()
lr_c = LogisticRegression(random_state=42, max_iter=1000)
lr_c.fit(sc_c.fit_transform(Xtr_c), ytr_c)
yp_c = lr_c.predict(sc_c.transform(Xte_c))

results_cons = df.loc[Xte_c.index].copy()
results_cons['predicted'] = yp_c
results_cons['correct'] = (results_cons['predicted'] == results_cons[TARGET]).astype(int)
results_cons['prior_tier'] = pd.cut(
    results_cons['prior_completion_rate'],
    bins=[0, 0.60, 0.75, 0.90, 1.01],
    labels=['<60%', '60–75%', '75–90%', '>90%']
)

print('=== Bias Comparison: Full Model vs. Conservative Model ===')
print()
print(f'{"Tier":<12} {"Full accuracy":>16} {"Cons accuracy":>16} {"Change":>10}')
print('-' * 56)

for tier in tier_order:
    full_sub = results[results['prior_tier'] == tier]
    cons_sub = results_cons[results_cons['prior_tier'] == tier]

    full_acc = full_sub['correct'].mean() if len(full_sub) > 0 else float('nan')
    cons_acc = cons_sub['correct'].mean() if len(cons_sub) > 0 else float('nan')
    change = cons_acc - full_acc

    print(f'{tier:<12} {full_acc*100:>14.1f}%  {cons_acc*100:>14.1f}%  {change*100:>+8.1f}pp')

print()
print(f'Overall — Full:         {accuracy_score(y_test, y_pred)*100:.1f}%')
print(f'Overall — Conservative: {accuracy_score(yte_c, yp_c)*100:.1f}%')

In [ ]:
# Visualize the accuracy comparison
full_accs  = [results[results['prior_tier']==t]['correct'].mean() * 100 for t in tier_order]
cons_accs  = [results_cons[results_cons['prior_tier']==t]['correct'].mean() * 100 for t in tier_order]

x = np.arange(len(tier_order))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.2, full_accs, width=0.35, label='Full model (with history)',
       color='#5C8BC7', alpha=0.9)
ax.bar(x + 0.2, cons_accs, width=0.35, label='Conservative model (no history)',
       color='#81C784', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(tier_order)
ax.set_xlabel('Prior Completion Rate Tier')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Does Removing Student History Reduce the Accuracy Gap?', fontsize=13, fontweight='bold')
ax.set_ylim(0, 120)
ax.legend()
for i, (f, c) in enumerate(zip(full_accs, cons_accs)):
    ax.text(i - 0.2, f + 1, f'{f:.0f}%', ha='center', fontsize=9)
    ax.text(i + 0.2, c + 1, f'{c:.0f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### Finding 5: Removing History Doesn't Fix the Bias — But It Does Change the Model

Removing `prior_completion_rate` and `prior_avg_grade` does not eliminate the accuracy disparity across tiers. The gap between high-history and low-history students is similar in both models.

This tells us something important: **the bias is not only in the features we chose — it is in the data itself.** Students with low prior completion rates are underrepresented in the training data. Even without the explicit history features, the model struggles with this group because it has so few examples to learn from.

This is a harder problem to solve than feature selection alone can address. Genuinely fair AI requires:
- More representative training data
- Techniques specifically designed to improve performance on minority groups
- Or, for some applications, honest acknowledgment that the AI should not be used for consequential decisions about underrepresented individuals

> 📝 **Reflection:** You have now found bias in the model that cannot be fully fixed by changing which features you use. What does this mean for whether the model should be deployed in a real school? Write a 3–4 sentence recommendation.

*[Write your recommendation here]*

---

## Part 7: Producing the Bias Audit Report

In [ ]:
# Formal bias audit report
print('=' * 65)
print('BIAS AUDIT REPORT — Module 4')
print('Project: Predicting On-Time Homework Submission')
print('=' * 65)
print()
print(f'Overall test accuracy:          {overall_acc*100:.1f}%')
print()

print('PERFORMANCE BY PRIOR COMPLETION RATE TIER:')
print(f'  {"Tier":<12} {"n":>5} {"Accuracy":>10} {"Late caught":>12} {"Late total":>12}')
print('  ' + '-' * 48)
for tier in tier_order:
    sub = results[results['prior_tier'] == tier]
    if len(sub) == 0: continue
    acc = sub['correct'].mean()
    late_total = (sub[TARGET] == 0).sum()
    late_caught = ((sub['predicted'] == 0) & (sub[TARGET] == 0)).sum()
    print(f'  {tier:<12} {len(sub):>5} {acc*100:>9.1f}%  {late_caught:>8}/{late_total:<8}')

print()
print('KEY FINDINGS:')
print('  1. Students with lowest prior completion rates have worst model performance')
print('  2. The model is effectively useless for students with <75% prior completion')
print('  3. These are exactly the students who most need early intervention')
print('  4. Root cause: underrepresentation in training data')
print('  5. Removing history features does not resolve the bias')
print()
print('FAIRNESS METRICS:')
accs = [results[results['prior_tier']==t]['correct'].mean() for t in tier_order
        if len(results[results['prior_tier']==t]) > 0]
print(f'  Max accuracy gap across tiers: {(max(accs)-min(accs))*100:.1f} percentage points')
print(f'  Tier with worst performance: 60–75% ({min(accs)*100:.1f}% accurate)')
print(f'  Tier with best performance:  >90%   ({max(accs)*100:.1f}% accurate)')
print()
print('RECOMMENDATION:')
print('  [Write your recommendation in the reflection cell above]')
print('=' * 65)

---

## Part 8: A Framework for Thinking About AI Fairness

The bias audit revealed a specific problem in a specific model. But the thinking tools we developed apply broadly. Before any AI system is deployed in a context that affects real people, it should be audited along these dimensions:

| Fairness Dimension | Question to Ask | How We Measured It |
|---|---|---|
| **Accuracy parity** | Is the model equally accurate across subgroups? | Accuracy by tier, type, difficulty |
| **Recall parity** | Does the model catch the target outcome equally across groups? | Late catch rate by tier |
| **Precision parity** | Does the model flag false positives equally across groups? | False flag rate by tier |
| **Representation** | Is every group adequately represented in training data? | Count by tier in training set |
| **Confidence calibration** | Is the model's confidence level trustworthy across groups? | Predicted probability distribution by tier |

### The Catholic Standard

*Magnifica Humanitas* (no. 109) states that to speak of social justice requires questioning the global distribution of power that decides who in fact can train these models and **who is merely subjected to them**. In the context of our homework model:

- Students with strong academic histories benefit from the model (it confirms them as "on time")
- Students with weaker histories are **subjected to a model that fails them** — it misses their late submissions, provides no useful signal, and may assign them incorrect predictions with high confidence

This is not an abstract philosophical concern. It is a measurable, documented pattern in the model you built. And it should change what you recommend about how — or whether — this model is deployed.

---

## Summary and What's Coming Next

### What You Accomplished in This Module

- Conducted a structured bias audit across assignment type, difficulty level, prior completion rate, and individual students
- Discovered a 33-percentage-point accuracy gap between the highest- and lowest-performing student tiers
- Found that the model is most inaccurate for the students who are most likely to be late and most in need of support
- Identified the root cause: underrepresentation of struggling students in the training data
- Tested whether removing history-based features reduces bias — and found it does not fully resolve it
- Produced a formal bias audit report and grounded the findings in *Magnifica Humanitas*

### Coming in Module 5 (Classes 9–10)

Module 5 addresses the governance question: **given everything we have found, under what conditions — if any — should this model be deployed?** You will draft a data governance policy for the system, specifying who should have access to predictions, what human oversight mechanisms must be in place, what disclosures students and families deserve, and what the conditions for revoking the system would be.

This will form the foundation of your final project presentation in Class 13.

---

> **A thought to carry forward:** *Magnifica Humanitas* (no. 102) says AI systems can cause injustice that "goes unnoticed" because it is cloaked in a veneer of neutrality. The bias you found in Module 4 would be invisible if you had only looked at the overall accuracy number. It took a deliberate audit — the decision to look — to find it. Most deployed AI systems have never been subjected to this kind of scrutiny. Who, in your view, is responsible for making sure they are?